# MiningSim Wireline Hole Inspection Workbench

<div style="padding:16px 18px;border:1px solid #d9e2ec;border-left:5px solid #1f6f78;border-radius:8px;background:#f7fafc;margin:10px 0 18px 0;">
<b>Purpose.</b> Prototype the ingestion, quality control, visualisation, reconciliation and engineering calculations required for a MiningSim web application that inspects wireline logs from drilled blast holes.
</div>

This notebook builds on the earlier gamma-focused work, but reads the **actual LAS curve definitions** rather than assuming a fixed column order. It also reads GeoVista-style XHD metadata and calibration records directly from ZIP archives, preserves XRD files for provenance, and provides reusable functions for:

- LAS/XHD archive inventory and validation;
- multi-track log inspection across gamma, caliper, temperature/conductivity and verticality tools;
- four-arm caliper calibration, cross-sections, hole volume and diameter reconciliation;
- borehole trajectory calculation from inclination and azimuth;
- repeated-run comparison and export-ready tables for a future web application.

> **Important engineering assumption:** the default caliper geometry treats `X1`, `X2`, `Y1`, `Y2` as independent radial arm distances and sums opposite arms to obtain the two orthogonal borehole diameters. The notebook also calculates the alternative “per-arm diameter mean” interpretation so the impact is visible. Confirm the tool convention with the manufacturer or calibration procedure before production use.

## 1. Scope, limitations and supplied-data findings

The supplied files use three companion formats:

- **LAS** — interpreted tabular log data and curve headers; this is the primary analytics input.
- **XHD** — XML metadata, sonde stack definitions, timestamps and calibration coefficients.
- **XRD** — proprietary binary acquisition data. It is inventoried and retained for traceability, but not decoded here because no public binary specification was supplied.

The notebook will verify these observations from the files themselves:

1. `Calibrated.las` and `Uncalibrated.las` contain four caliper channels: `X1`, `X2`, `Y1`, `Y2` in millimetres.
2. `Calibrated.xhd` contains four three-point polynomial calibrations using 150, 250 and 350 mm reference values.
3. `ACS-03_Run10_10m_min_up` has XHD/XRD companions but no LAS export.
4. The LAS `DATE` strings are malformed. They match `year-minute-day` from the XHD timestamp, so XHD `LogCreated` is treated as canonical.

## 2. Imports, display theme and configuration

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path, PurePosixPath
from typing import Any, Iterable, Mapping, Optional, Sequence
from zipfile import ZipFile
import io
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

MS_COLOURS = ['#0B3C5D', '#1D70A2', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51', '#6C5B7B', '#59636E']

# Search locations. Add project folders here when the notebook is moved out of this workspace.
SEARCH_ROOTS = [Path.cwd(), Path('/mnt/data')]
EXPLICIT_INPUTS: list[Path] = []

# Engineering inputs — replace these with the hole plan / inspection register.
NOMINAL_DIAMETER_MM = 150.0
DIAMETER_TOLERANCE_MM = 10.0
CALIPER_GEOMETRY_MODE = 'opposite_arm_sum'  # alternative: 'per_arm_diameter_mean'
CALIPER_MOTOR_CURRENT_MIN_MA = 1.8
CALIPER_MAX_ARM_MM = 1_000.0
TRAJECTORY_INCLINATION_REFERENCE = 'vertical'  # alternative: 'horizontal'

EXPORT_DIR = Path('./miningsim_exports')

HTML_STYLE = """
<style>
.jp-RenderedHTMLCommon table {font-size: 12px;}
.ms-card {display:inline-block;vertical-align:top;min-width:180px;margin:4px 8px 8px 0;padding:12px 14px;border:1px solid #d9e2ec;border-radius:8px;background:#ffffff;box-shadow:0 1px 2px rgba(0,0,0,.04)}
.ms-card .label {font-size:11px;text-transform:uppercase;letter-spacing:.04em;color:#59636e}
.ms-card .value {font-size:22px;font-weight:700;color:#0b3c5d;margin-top:4px}
.ms-callout {padding:13px 15px;border:1px solid #d9e2ec;border-left:5px solid #1f6f78;border-radius:8px;background:#f7fafc;margin:10px 0}
.ms-warning {padding:13px 15px;border:1px solid #f1d3a2;border-left:5px solid #e9a23b;border-radius:8px;background:#fffaf0;margin:10px 0}
.ms-ok {padding:13px 15px;border:1px solid #b7dfcf;border-left:5px solid #2a9d8f;border-radius:8px;background:#f3fbf8;margin:10px 0}
</style>
"""
display(HTML(HTML_STYLE))
print('Notebook environment ready.')

Notebook environment ready.


## 3. Data model and LAS/XHD parsers

In [2]:
@dataclass
class CurveMeta:
    column: str
    mnemonic: str
    unit: str
    description: str
    ordinal: int
    sonde_name: Optional[str] = None
    sonde_serial: Optional[int] = None
    receiver_offset_m: Optional[float] = None


@dataclass
class CalibrationRecord:
    sonde_name: str
    sonde_serial: Optional[int]
    sonde_id: Optional[int]
    channel_name: str
    channel_index: int
    mode: str
    coefficients: tuple[float, float, float, float]
    raw_values: tuple[float, float, float, float]
    new_values: tuple[float, float, float, float]
    calibration_date: Optional[str]
    name_inferred: bool = False


@dataclass
class LogRecord:
    log_id: str
    archive_path: Path
    las_member: str
    xhd_member: Optional[str]
    xrd_member: Optional[str]
    df: pd.DataFrame
    curves: list[CurveMeta]
    version: dict[str, dict[str, str]]
    well: dict[str, dict[str, str]]
    parameters: dict[str, dict[str, str]]
    other: list[str]
    rejected_rows: int = 0
    xhd: dict[str, Any] = field(default_factory=dict)
    calibrations: list[CalibrationRecord] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    @property
    def index_unit(self) -> str:
        return self.curves[0].unit if self.curves else ''

    @property
    def view_type(self) -> Optional[str]:
        return self.xhd.get('view_type')

    @property
    def log_created(self) -> Optional[pd.Timestamp]:
        value = self.xhd.get('log_created')
        return pd.Timestamp(value) if value else None

    def curve(self, mnemonic: str, sonde_contains: Optional[str] = None, occurrence: int = 0) -> Optional[CurveMeta]:
        key = _normalise_name(mnemonic)
        matches = [curve for curve in self.curves if _normalise_name(curve.mnemonic) == key]
        if sonde_contains:
            sonde_key = sonde_contains.casefold()
            matches = [curve for curve in matches if curve.sonde_name and sonde_key in curve.sonde_name.casefold()]
        return matches[occurrence] if 0 <= occurrence < len(matches) else None


def _normalise_name(value: Any) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


def _parse_las_definition(line: str) -> Optional[dict[str, str]]:
    left, _, description = line.partition(':')
    match = re.match(r'^([^\.\s]+)\.([^\s]*)\s*(.*)$', left.strip())
    if not match:
        return None
    mnemonic, unit, value = match.groups()
    return {
        'mnemonic': mnemonic.strip(),
        'unit': unit.strip(),
        'value': value.strip(),
        'description': description.strip(),
    }


def parse_las_bytes(payload: bytes, source_name: str) -> dict[str, Any]:
    """Parse an unwrapped LAS 2.x file while preserving its declared curve order and units."""
    text = payload.decode('utf-8-sig', errors='replace')
    lines = text.splitlines()

    sections: dict[str, list[str]] = {}
    section_name = ''
    ascii_start: Optional[int] = None

    for index, raw_line in enumerate(lines):
        stripped = raw_line.strip()
        if stripped.startswith('~'):
            section_name = stripped.upper()
            if section_name.startswith('~ASCII'):
                ascii_start = index
                break
            sections.setdefault(section_name, [])
            continue
        if section_name:
            sections.setdefault(section_name, []).append(raw_line)

    if ascii_start is None:
        raise ValueError(f'{source_name}: LAS ASCII data section not found.')

    def section(prefix: str) -> list[str]:
        for key, values in sections.items():
            if key.startswith(prefix):
                return values
        return []

    def parse_named_section(prefix: str) -> dict[str, dict[str, str]]:
        output: dict[str, dict[str, str]] = {}
        for line in section(prefix):
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                continue
            parsed = _parse_las_definition(stripped)
            if parsed:
                output[parsed['mnemonic'].upper()] = parsed
        return output

    version = parse_named_section('~VERSION')
    well = parse_named_section('~WELL')
    parameters = parse_named_section('~PARAMETER')

    wrap_value = version.get('WRAP', {}).get('value', 'NO').upper()
    if wrap_value not in {'NO', 'N'}:
        raise NotImplementedError(f'{source_name}: WRAP={wrap_value!r} is not supported by this prototype parser.')

    curve_definitions: list[dict[str, str]] = []
    for line in section('~CURVE'):
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        parsed = _parse_las_definition(stripped)
        if parsed:
            curve_definitions.append(parsed)

    if not curve_definitions:
        raise ValueError(f'{source_name}: no curve definitions were found.')

    occurrence_count: dict[str, int] = {}
    curves: list[CurveMeta] = []
    for ordinal, item in enumerate(curve_definitions):
        mnemonic = item['mnemonic']
        occurrence_count[mnemonic] = occurrence_count.get(mnemonic, 0) + 1
        suffix = occurrence_count[mnemonic]
        column = mnemonic if suffix == 1 else f'{mnemonic}__{suffix}'
        curves.append(CurveMeta(
            column=column,
            mnemonic=mnemonic,
            unit=item['unit'],
            description=item['description'],
            ordinal=ordinal,
        ))

    rows: list[np.ndarray] = []
    rejected_rows = 0
    expected_columns = len(curves)
    for line in lines[ascii_start + 1:]:
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        values = np.fromstring(stripped, sep=' ')
        if values.size != expected_columns:
            rejected_rows += 1
            continue
        rows.append(values)

    if not rows:
        raise ValueError(f'{source_name}: no valid numeric rows were found.')

    frame = pd.DataFrame(np.vstack(rows), columns=[curve.column for curve in curves])
    null_text = well.get('NULL', {}).get('value', '-999.25').split()[0]
    try:
        null_value = float(null_text)
        frame = frame.mask(np.isclose(frame, null_value, equal_nan=False))
    except ValueError:
        null_value = np.nan

    other = [
        line.strip() for line in section('~OTHER')
        if line.strip() and not line.strip().startswith('#')
    ]

    return {
        'df': frame,
        'curves': curves,
        'version': version,
        'well': well,
        'parameters': parameters,
        'other': other,
        'rejected_rows': rejected_rows,
        'null_value': null_value,
    }

In [3]:
def _safe_int(value: Optional[str]) -> Optional[int]:
    try:
        return int(value) if value not in (None, '') else None
    except (TypeError, ValueError):
        return None


def _safe_float(value: Optional[str]) -> Optional[float]:
    try:
        return float(value) if value not in (None, '') else None
    except (TypeError, ValueError):
        return None


def _float_tuple(parent: ET.Element, path: str) -> tuple[float, float, float, float]:
    values = [_safe_float(node.text) or 0.0 for node in parent.findall(path)]
    values = (values + [0.0, 0.0, 0.0, 0.0])[:4]
    return tuple(values)  # type: ignore[return-value]


def parse_xhd_bytes(payload: bytes) -> dict[str, Any]:
    """Parse the GeoVista XML header and calibration records."""
    root = ET.fromstring(payload.decode('utf-8-sig', errors='replace'))
    nil_attr = '{http://www.w3.org/2001/XMLSchema-instance}nil'

    sonde_definitions: dict[int, dict[str, Any]] = {}
    for definition in root.findall('./Sondes/SondeDefinition'):
        sonde_id = _safe_int(definition.findtext('SondeID'))
        if sonde_id is None:
            continue
        channels = []
        for channel in definition.findall('.//ChannelList/SondeChannel'):
            channels.append({
                'channel_name': channel.findtext('ChannelName') or '',
                'channel_id': _safe_int(channel.findtext('ChannelID')),
                'units': channel.findtext('Units') or '',
                'receiver_offset_m': _safe_float(channel.findtext('ReceiverOffset')),
            })
        sonde_definitions[sonde_id] = {
            'sonde_id': sonde_id,
            'sonde_name': definition.findtext('SondeName') or f'Sonde {sonde_id}',
            'channels': channels,
        }

    selected_ids = [_safe_int(node.text) or 0 for node in root.findall('./SelectedStackDefinition/SondeID/int')]
    selected_serials = [_safe_int(node.text) or 0 for node in root.findall('./SelectedStackDefinition/SondeSerial/int')]
    selected_enabled = [(node.text or '').strip().casefold() == 'true' for node in root.findall('./SelectedStackDefinition/IsEnabled/boolean')]
    selected_stack = []
    for index, sonde_id in enumerate(selected_ids):
        if sonde_id == 0:
            continue
        enabled = selected_enabled[index] if index < len(selected_enabled) else True
        if not enabled:
            continue
        definition = sonde_definitions.get(sonde_id, {'sonde_name': f'Sonde {sonde_id}', 'channels': []})
        selected_stack.append({
            'sonde_id': sonde_id,
            'sonde_serial': selected_serials[index] if index < len(selected_serials) else None,
            'sonde_name': definition['sonde_name'],
            'channels': definition['channels'],
        })

    calibrations: list[CalibrationRecord] = []
    for sonde_cal in root.findall('./Calibration/SondeCalibration'):
        if sonde_cal.get(nil_attr) == 'true':
            continue
        sonde_name = sonde_cal.findtext('SondeName') or ''
        sonde_serial = _safe_int(sonde_cal.findtext('SondeSerial'))
        sonde_id = _safe_int(sonde_cal.findtext('SondeID'))
        calibration_date = sonde_cal.findtext('CalibrationDate')
        channel_definitions = sonde_definitions.get(sonde_id or -1, {}).get('channels', [])

        for channel_index, calibration in enumerate(sonde_cal.findall('.//CalibrationData')):
            explicit_name = (calibration.findtext('ChannelName') or '').strip()
            inferred_name = ''
            if not explicit_name and channel_index < len(channel_definitions):
                inferred_name = channel_definitions[channel_index].get('channel_name', '')
            channel_name = explicit_name or inferred_name or f'channel_{channel_index}'
            calibrations.append(CalibrationRecord(
                sonde_name=sonde_name,
                sonde_serial=sonde_serial,
                sonde_id=sonde_id,
                channel_name=channel_name,
                channel_index=channel_index,
                mode=calibration.findtext('Mode') or 'None',
                coefficients=_float_tuple(calibration, './Coefficients/decimal'),
                raw_values=_float_tuple(calibration, './RawDataValues/decimal'),
                new_values=_float_tuple(calibration, './NewValues/decimal'),
                calibration_date=calibration_date,
                name_inferred=not bool(explicit_name) and bool(inferred_name),
            ))

    return {
        'software_version': root.findtext('SoftwareVersion'),
        'log_created': root.findtext('LogCreated'),
        'view_type': root.findtext('ViewType'),
        'stack_length_m': _safe_float(root.findtext('StackLength')),
        'selected_stack': selected_stack,
        'sonde_definitions': sonde_definitions,
        'calibrations': calibrations,
        'general_view': {
            'start_depth_m': _safe_float(root.findtext('./GeneralView/StartDepth')),
            'end_depth_m': _safe_float(root.findtext('./GeneralView/EndDepth')),
            'depth_when_stopped_m': _safe_float(root.findtext('./GeneralView/DepthWhenLogStopped')),
        },
    }


def assign_curve_owners(curves: list[CurveMeta], xhd: Mapping[str, Any]) -> list[CurveMeta]:
    """Attach sonde ownership by matching the reversed selected stack to the LAS curve sequence."""
    output = [replace(curve) for curve in curves]
    current_positions = [i for i, curve in enumerate(output) if _normalise_name(curve.mnemonic) == 'current']
    start = current_positions[0] + 1 if current_positions else 0

    flattened: list[dict[str, Any]] = []
    for sonde in reversed(list(xhd.get('selected_stack', []))):
        for channel in sonde.get('channels', []):
            flattened.append({**channel, **{
                'sonde_name': sonde.get('sonde_name'),
                'sonde_serial': sonde.get('sonde_serial'),
            }})

    for curve, channel in zip(output[start:], flattened):
        if _normalise_name(curve.mnemonic) != _normalise_name(channel.get('channel_name', '')):
            # Do not force a potentially incorrect ownership assignment.
            continue
        curve.sonde_name = channel.get('sonde_name')
        curve.sonde_serial = channel.get('sonde_serial')
        curve.receiver_offset_m = channel.get('receiver_offset_m')
        if not curve.unit and channel.get('units'):
            curve.unit = channel['units']
    return output

## 4. Discover and load the supplied archives

In [4]:
def discover_zip_inputs(search_roots: Sequence[Path], explicit: Sequence[Path] = ()) -> list[Path]:
    found: dict[str, Path] = {}
    for item in explicit:
        path = Path(item).expanduser().resolve()
        if path.is_file() and path.suffix.casefold() == '.zip':
            found[str(path)] = path
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.glob('*.zip'):
            found[str(path.resolve())] = path.resolve()
    return sorted(found.values(), key=lambda path: path.name.casefold())


def _companion_name(member: str, suffix: str) -> str:
    posix = PurePosixPath(member)
    return str(posix.with_suffix(suffix))


def load_zip_archives(paths: Sequence[Path]) -> tuple[dict[str, LogRecord], pd.DataFrame]:
    logs: dict[str, LogRecord] = {}
    inventory_rows: list[dict[str, Any]] = []

    for archive_path in paths:
        with ZipFile(archive_path) as archive:
            members = [name for name in archive.namelist() if not name.endswith('/')]
            grouped: dict[str, set[str]] = {}
            for member in members:
                suffix = PurePosixPath(member).suffix.casefold()
                if suffix not in {'.las', '.xhd', '.xrd'}:
                    continue
                stem = str(PurePosixPath(member).with_suffix(''))
                grouped.setdefault(stem, set()).add(suffix)

            for stem, suffixes in sorted(grouped.items()):
                inventory_rows.append({
                    'archive': archive_path.name,
                    'item': PurePosixPath(stem).name,
                    'LAS': '.las' in suffixes,
                    'XHD': '.xhd' in suffixes,
                    'XRD': '.xrd' in suffixes,
                    'complete_triplet': {'.las', '.xhd', '.xrd'}.issubset(suffixes),
                })

            for las_member in sorted(name for name in members if name.casefold().endswith('.las')):
                parsed = parse_las_bytes(archive.read(las_member), f'{archive_path.name}:{las_member}')
                xhd_member = _companion_name(las_member, '.xhd')
                xrd_member = _companion_name(las_member, '.xrd')
                xhd = parse_xhd_bytes(archive.read(xhd_member)) if xhd_member in members else {}
                curves = assign_curve_owners(parsed['curves'], xhd) if xhd else parsed['curves']
                log_id = PurePosixPath(las_member).stem
                unique_id = log_id
                if unique_id in logs:
                    unique_id = f'{archive_path.stem}::{log_id}'

                warnings_list: list[str] = []
                raw_las_date = parsed['well'].get('DATE', {}).get('value')
                log_created = xhd.get('log_created')
                if raw_las_date and log_created:
                    try:
                        stamp = pd.Timestamp(log_created)
                        logger_bug_value = f'{stamp.year:04d}-{stamp.minute:02d}-{stamp.day:02d}'
                        if raw_las_date == logger_bug_value:
                            warnings_list.append('LAS DATE encodes year-minute-day; XHD LogCreated is canonical.')
                    except Exception:
                        pass

                logs[unique_id] = LogRecord(
                    log_id=unique_id,
                    archive_path=archive_path,
                    las_member=las_member,
                    xhd_member=xhd_member if xhd_member in members else None,
                    xrd_member=xrd_member if xrd_member in members else None,
                    df=parsed['df'],
                    curves=curves,
                    version=parsed['version'],
                    well=parsed['well'],
                    parameters=parsed['parameters'],
                    other=parsed['other'],
                    rejected_rows=parsed['rejected_rows'],
                    xhd=xhd,
                    calibrations=list(xhd.get('calibrations', [])),
                    warnings=warnings_list,
                )

    return logs, pd.DataFrame(inventory_rows)


def classify_log(log: LogRecord) -> str:
    names = {_normalise_name(curve.mnemonic) for curve in log.curves}
    tags: list[str] = []
    if {'x1', 'x2', 'y1', 'y2'}.issubset(names):
        tags.append('4-arm caliper')
    if 'gr' in names or 'api' in names:
        tags.append('gamma')
    if {'incline', 'azimuth'}.issubset(names):
        tags.append('directional')
    if 'conductivity' in names or 'pt100' in names:
        tags.append('temperature/conductivity')
    if log.index_unit.casefold().startswith('second') or (log.view_type or '').casefold() == 'time':
        tags.insert(0, 'telemetry/time-series')
    return ' + '.join(dict.fromkeys(tags)) or 'generic LAS'


def physical_depth_column(log: LogRecord) -> str:
    if log.index_unit.casefold().startswith('second'):
        return log.curves[0].column
    candidate = log.curve('Depth')
    if candidate and log.df[candidate.column].notna().sum() >= 2:
        return candidate.column
    return log.curves[0].column


def curve_unit(log: LogRecord, column: str) -> str:
    match = next((curve for curve in log.curves if curve.column == column), None)
    return match.unit if match else ''


ZIP_INPUTS = discover_zip_inputs(SEARCH_ROOTS, EXPLICIT_INPUTS)
if not ZIP_INPUTS:
    raise FileNotFoundError('No ZIP archives were found. Set EXPLICIT_INPUTS or add a search root.')

print('Discovered archives:')
for path in ZIP_INPUTS:
    print(f'  • {path}')

logs, archive_inventory = load_zip_archives(ZIP_INPUTS)
print(f'\nLoaded {len(logs)} LAS logs from {len(ZIP_INPUTS)} archives.')

Discovered archives:
  • /mnt/data/ACS03 Calibration runs(1).zip
  • /mnt/data/Caliper(1).zip



Loaded 13 LAS logs from 2 archives.


In [5]:
def log_summary_table(logs: Mapping[str, LogRecord]) -> pd.DataFrame:
    rows = []
    for log_id, log in logs.items():
        depth_col = physical_depth_column(log)
        depth = pd.to_numeric(log.df[depth_col], errors='coerce').dropna()
        raw_date = log.well.get('DATE', {}).get('value')
        median_step = float(np.nanmedian(np.abs(np.diff(depth)))) if len(depth) > 1 else np.nan
        duplicate_mnemonics = pd.Series([curve.mnemonic for curve in log.curves]).duplicated().sum()
        rows.append({
            'log_id': log_id,
            'archive': log.archive_path.name,
            'classification': classify_log(log),
            'view': log.view_type,
            'created_from_xhd': str(log.log_created) if log.log_created is not None else None,
            'LAS_DATE_raw': raw_date,
            'rows': len(log.df),
            'curves': len(log.curves),
            'index_unit': log.index_unit,
            'depth_min': float(depth.min()) if not depth.empty else np.nan,
            'depth_max': float(depth.max()) if not depth.empty else np.nan,
            'median_sample_step': median_step,
            'duplicate_mnemonics': int(duplicate_mnemonics),
            'active_calibrations': sum(record.mode != 'None' for record in log.calibrations),
            'rejected_rows': log.rejected_rows,
            'warnings': ' | '.join(log.warnings),
        })
    return pd.DataFrame(rows).sort_values(['archive', 'log_id']).reset_index(drop=True)

summary_table = log_summary_table(logs)
display(summary_table)

incomplete = archive_inventory.loc[~archive_inventory['complete_triplet']].copy()
if not incomplete.empty:
    display(HTML('<div class="ms-warning"><b>Incomplete companion sets found.</b> The items below cannot be fully processed as LAS logs.</div>'))
    display(incomplete)
else:
    display(HTML('<div class="ms-ok"><b>All archive items contain LAS, XHD and XRD companions.</b></div>'))

,log_id,archive,classification,view,created_from_xhd,LAS_DATE_raw,rows,curves,index_unit,depth_min,depth_max,median_sample_step,duplicate_mnemonics,active_calibrations,rejected_rows,warnings
0,ACS-03_Run1,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:13:13.536594300+08:00,2026-13-09,2713,32,Metres,-2.1200,25.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
1,ACS-03_Run10_10m_min_down,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Down,2026-07-09 17:02:41.103266+08:00,2026-02-09,2046,32,Metres,2.0000,22.4500,0.0100,1,17,0,LAS DATE encodes year-minute-day; XHD LogCreat...
2,ACS-03_Run2,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:23:10.798329400+08:00,2026-23-09,2806,32,Metres,-2.0500,26.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
3,ACS-03_Run3,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:29:57.488736900+08:00,2026-29-09,2806,32,Metres,-2.0500,26.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
4,ACS-03_Run4,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:36:21.794934800+08:00,2026-36-09,2805,32,Metres,-2.0400,26.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
5,ACS-03_Run5,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:40:20.937837+08:00,2026-40-09,2806,32,Metres,-2.0500,26.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
6,ACS-03_Run6,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:45:32.863937800+08:00,2026-45-09,2506,32,Metres,-2.0500,23.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
7,ACS-03_Run7,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 15:48:40.451907900+08:00,2026-48-09,2806,32,Metres,-2.0500,26.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
8,ACS-03_Run8,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Down,2026-07-09 15:52:41.203715300+08:00,2026-52-09,770,32,Metres,17.0000,24.6900,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...
9,ACS-03_Run9_5m_min_up,ACS03 Calibration runs(1).zip,gamma + directional + temperature/conductivity,Up,2026-07-09 16:03:06.568167600+08:00,2026-03-09,2106,32,Metres,-2.0500,19.0000,0.0100,1,15,0,LAS DATE encodes year-minute-day; XHD LogCreat...


,archive,item,LAS,XHD,XRD,complete_triplet
2,ACS03 Calibration runs(1).zip,ACS-03_Run10_10m_min_up,False,True,True,False


## 5. Confirm the caliper content and calibration records

In [6]:
ARM_NAMES = ('X1', 'X2', 'Y1', 'Y2')


def is_caliper_log(log: LogRecord) -> bool:
    return all(log.curve(name) is not None for name in ARM_NAMES)


def caliper_calibration_table(log: LogRecord) -> pd.DataFrame:
    rows = []
    for record in log.calibrations:
        if 'caliper' not in record.sonde_name.casefold() or record.channel_name not in ARM_NAMES:
            continue
        rows.append({
            'channel': record.channel_name,
            'mode': record.mode,
            'c0': record.coefficients[0],
            'c1': record.coefficients[1],
            'c2': record.coefficients[2],
            'c3': record.coefficients[3],
            'raw_point_1': record.raw_values[0],
            'raw_point_2': record.raw_values[1],
            'raw_point_3': record.raw_values[2],
            'reference_mm_1': record.new_values[0],
            'reference_mm_2': record.new_values[1],
            'reference_mm_3': record.new_values[2],
            'calibration_date': record.calibration_date,
            'channel_name_inferred_from_order': record.name_inferred,
        })
    return pd.DataFrame(rows)

caliper_ids = [log_id for log_id, log in logs.items() if is_caliper_log(log)]
if not caliper_ids:
    raise AssertionError('No four-arm caliper log was detected.')

print('Caliper logs:', caliper_ids)
for log_id in caliper_ids:
    log = logs[log_id]
    curves = [curve for curve in log.curves if curve.mnemonic in ARM_NAMES]
    display(pd.DataFrame([{
        'log_id': log_id,
        'column': curve.column,
        'mnemonic': curve.mnemonic,
        'unit': curve.unit,
        'sonde': curve.sonde_name,
        'serial': curve.sonde_serial,
        'receiver_offset_m': curve.receiver_offset_m,
        'non_null_samples': int(log.df[curve.column].notna().sum()),
        'median': float(log.df[curve.column].median()),
        'maximum': float(log.df[curve.column].max()),
    } for curve in curves]))

calibrated_id = next((item for item in caliper_ids if item.casefold() == 'calibrated'), caliper_ids[0])
uncalibrated_id = next((item for item in caliper_ids if 'uncalibrated' in item.casefold()), None)
calibrated_log = logs[calibrated_id]
uncalibrated_log = logs[uncalibrated_id] if uncalibrated_id else None

cal_table = caliper_calibration_table(calibrated_log)
display(HTML('<div class="ms-ok"><b>Four caliper curves found.</b> The calibrated XHD also contains the following polynomial records.</div>'))
display(cal_table)

Caliper logs: ['Calibrated', 'Uncalibrated']


,log_id,column,mnemonic,unit,sonde,serial,receiver_offset_m,non_null_samples,median,maximum
0,Calibrated,X1,X1,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,61.4239,61.4818
1,Calibrated,X2,X2,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,67.6801,67.7284
2,Calibrated,Y1,Y1,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,79.8488,80.7513
3,Calibrated,Y2,Y2,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,72.7597,72.8141


,log_id,column,mnemonic,unit,sonde,serial,receiver_offset_m,non_null_samples,median,maximum
0,Uncalibrated,X1,X1,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,"3,780,178.0000","3,806,522.0000"
1,Uncalibrated,X2,X2,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,"3,483,684.8000","3,484,446.0000"
2,Uncalibrated,Y1,Y1,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,"3,389,758.8000","3,391,644.0000"
3,Uncalibrated,Y2,Y2,mm,Caliper Sonde (4 Arm),6341,0.3000,2617,"4,203,983.2000","4,215,194.0000"


,channel,mode,c0,c1,c2,c3,raw_point_1,raw_point_2,raw_point_3,reference_mm_1,reference_mm_2,reference_mm_3,calibration_date,channel_name_inferred_from_order
0,X1,ThreePoint,-191.9989,0.0001,-0.0000,0.0000,"5,245,529.4000","6,980,659.4000","8,841,639.4000",150.0000,250.0000,350.0000,2026-06-05T11:45:56.5751612+08:00,True
1,X2,ThreePoint,-239.4175,0.0001,-0.0000,0.0000,"4,596,920.6000","6,143,118.6000","8,010,922.2000",150.0000,250.0000,350.0000,2026-06-05T11:45:56.5751612+08:00,True
2,Y1,ThreePoint,-161.4158,0.0001,-0.0000,0.0000,"4,646,991.6000","6,353,899.4000","8,210,634.4000",150.0000,250.0000,350.0000,2026-06-05T11:45:56.5751612+08:00,True
3,Y2,ThreePoint,-203.3433,0.0001,-0.0000,0.0000,"5,346,244.8000","7,050,710.8000","8,870,228.8000",150.0000,250.0000,350.0000,2026-06-05T11:45:56.5751612+08:00,True


### Interpretation of the two caliper LAS files

`Calibrated.las` contains values on a physical millimetre scale. `Uncalibrated.las` contains raw counts in the millions. The calibration coefficients are stored only in `Calibrated.xhd`; their channel names are blank in the XML, so the parser maps them to `X1`, `X2`, `Y1`, `Y2` by the declared caliper channel order and marks that mapping as inferred.

## 6. Quality-control functions and curve audit

In [7]:
def curve_audit(log: LogRecord) -> pd.DataFrame:
    rows = []
    for curve in log.curves:
        series = pd.to_numeric(log.df[curve.column], errors='coerce')
        valid = series.replace([np.inf, -np.inf], np.nan).dropna()
        flag = ''
        name = _normalise_name(curve.mnemonic)
        if not valid.empty:
            if name == 'temperature' and ((valid < -100).any() or (valid > 250).any()):
                flag = 'outside typical physical temperature range; may be raw/invalid'
            elif name == 'incline' and ((valid < 0).any() or (valid > 180).any()):
                flag = 'outside 0–180°'
            elif name == 'azimuth' and ((valid < 0).any() or (valid > 360).any()):
                flag = 'outside 0–360°'
            elif name in {'x1', 'x2', 'y1', 'y2'} and valid.median() > 10_000:
                flag = 'raw counts rather than calibrated millimetres'
        rows.append({
            'column': curve.column,
            'mnemonic': curve.mnemonic,
            'sonde': curve.sonde_name,
            'serial': curve.sonde_serial,
            'unit': curve.unit,
            'receiver_offset_m': curve.receiver_offset_m,
            'valid_samples': int(valid.size),
            'missing_pct': 100.0 * (1.0 - valid.size / max(len(series), 1)),
            'minimum': float(valid.min()) if not valid.empty else np.nan,
            'median': float(valid.median()) if not valid.empty else np.nan,
            'maximum': float(valid.max()) if not valid.empty else np.nan,
            'QC_flag': flag,
        })
    return pd.DataFrame(rows)


def log_qc_issues(log: LogRecord) -> list[str]:
    issues = list(log.warnings)
    if log.rejected_rows:
        issues.append(f'{log.rejected_rows} ASCII rows did not match the declared curve count.')
    depth_col = physical_depth_column(log)
    depth = pd.to_numeric(log.df[depth_col], errors='coerce').dropna()
    if len(depth) > 2:
        diffs = np.diff(depth)
        direction = np.sign(np.nanmedian(diffs))
        reversals = int(np.sum(np.sign(diffs[np.abs(diffs) > 1e-9]) != direction))
        if reversals:
            issues.append(f'{reversals} depth-direction reversals detected.')
    audit = curve_audit(log)
    flagged = audit.loc[audit['QC_flag'].ne(''), ['column', 'QC_flag']]
    for row in flagged.itertuples(index=False):
        issues.append(f'{row.column}: {row.QC_flag}.')
    return issues

qc_rows = []
for log_id, log in logs.items():
    issues = log_qc_issues(log)
    qc_rows.append({'log_id': log_id, 'issue_count': len(issues), 'issues': ' | '.join(issues)})
qc_summary = pd.DataFrame(qc_rows).sort_values(['issue_count', 'log_id'], ascending=[False, True])
display(qc_summary)

# Inspect the most complete directional run and the two caliper logs in detail.
for audit_id in [item for item in ['ACS-03_Run10_10m_min_down', calibrated_id, uncalibrated_id] if item and item in logs]:
    display(Markdown(f'**Curve audit — `{audit_id}`**'))
    display(curve_audit(logs[audit_id]))

,log_id,issue_count,issues
12,Uncalibrated,5,LAS DATE encodes year-minute-day; XHD LogCreat...
0,ACS-03_Run1,2,LAS DATE encodes year-minute-day; XHD LogCreat...
2,ACS-03_Run2,2,LAS DATE encodes year-minute-day; XHD LogCreat...
3,ACS-03_Run3,2,LAS DATE encodes year-minute-day; XHD LogCreat...
4,ACS-03_Run4,2,LAS DATE encodes year-minute-day; XHD LogCreat...
5,ACS-03_Run5,2,LAS DATE encodes year-minute-day; XHD LogCreat...
6,ACS-03_Run6,2,LAS DATE encodes year-minute-day; XHD LogCreat...
7,ACS-03_Run7,2,LAS DATE encodes year-minute-day; XHD LogCreat...
8,ACS-03_Run8,2,LAS DATE encodes year-minute-day; XHD LogCreat...
9,ACS-03_Run9_5m_min_up,2,LAS DATE encodes year-minute-day; XHD LogCreat...


**Curve audit — `ACS-03_Run10_10m_min_down`**

,column,mnemonic,sonde,serial,unit,receiver_offset_m,valid_samples,missing_pct,minimum,median,maximum,QC_flag
0,DEPTH,DEPTH,None,NaN,Metres,NaN,2046,0.0000,2.0000,12.2250,22.4500,
1,Depth,Depth,None,NaN,,NaN,2003,2.1017,2.4400,12.4408,22.4591,
2,Tension,Tension,None,NaN,,NaN,2003,2.1017,0.0000,0.0000,0.0000,
3,Time,Time,None,NaN,,NaN,2003,2.1017,6.8000,73.7000,133.7000,
4,Speed,Speed,None,NaN,,NaN,2003,2.1017,0.0000,10.0100,10.1500,
5,Voltage,Voltage,None,NaN,,NaN,2003,2.1017,0.0960,0.0960,0.0960,
6,Current,Current,None,NaN,,NaN,2003,2.1017,0.1020,0.1040,0.1040,
7,WET,WET,WLIS,"10,024.0000",,0.0500,2002,2.1505,0.0000,1.0000,1.0000,
8,Sig,Sig,WLIS,"10,024.0000",,0.0500,2002,2.1505,3.0000,32.5000,975.0000,
9,Vth,Vth,WLIS,"10,024.0000",,0.2000,2002,2.1505,25.0000,100.0000,100.0000,


**Curve audit — `Calibrated`**

,column,mnemonic,sonde,serial,unit,receiver_offset_m,valid_samples,missing_pct,minimum,median,maximum,QC_flag
0,DEPTH,DEPTH,None,NaN,Metres,NaN,2759,0.0000,0.4200,14.2100,28.0000,
1,Depth,Depth,None,NaN,,NaN,2588,6.1979,2.1269,15.0665,27.9979,
2,Tension,Tension,None,NaN,,NaN,2588,6.1979,"32,282.0000","32,967.0000","33,702.0000",
3,Time,Time,None,NaN,,NaN,2588,6.1979,108.3500,205.3750,302.6000,
4,Speed,Speed,None,NaN,,NaN,2584,6.3429,0.9950,7.9850,8.2900,
5,Voltage,Voltage,None,NaN,,NaN,2588,6.1979,0.0970,0.0970,0.0970,
6,Current,Current,None,NaN,,NaN,2588,6.1979,0.1000,0.1020,0.1040,
7,X1,X1,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.1468,9.6721,61.4239,61.4818,
8,X2,X2,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.1468,9.1415,67.6801,67.7284,
9,Y1,Y1,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.1468,11.6682,79.8488,80.7513,


**Curve audit — `Uncalibrated`**

,column,mnemonic,sonde,serial,unit,receiver_offset_m,valid_samples,missing_pct,minimum,median,maximum,QC_flag
0,DEPTH,DEPTH,None,NaN,Metres,NaN,2761,0.0000,0.4000,14.2000,28.0000,
1,Depth,Depth,None,NaN,,NaN,2590,6.1934,2.1041,15.0555,27.9994,
2,Tension,Tension,None,NaN,,NaN,2590,6.1934,"36,077.0000","37,074.0000","38,197.0000",
3,Time,Time,None,NaN,,NaN,2590,6.1934,148.6000,226.2750,304.0500,
4,Speed,Speed,None,NaN,,NaN,2589,6.2296,0.9890,9.9800,10.2200,
5,Voltage,Voltage,None,NaN,,NaN,2590,6.1934,0.0970,0.0970,0.0970,
6,Current,Current,None,NaN,,NaN,2590,6.1934,0.1000,0.1020,0.1040,
7,X1,X1,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.2155,"677,046.8000","3,780,178.0000","3,806,522.0000",raw counts rather than calibrated millimetres
8,X2,X2,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.2155,"628,527.2000","3,483,684.8000","3,484,446.0000",raw counts rather than calibrated millimetres
9,Y1,Y1,Caliper Sonde (4 Arm),"6,341.0000",mm,0.3000,2617,5.2155,"605,439.6000","3,389,758.8000","3,391,644.0000",raw counts rather than calibrated millimetres


## 7. Reusable multi-track log viewer

In [8]:
def resolve_track(log: LogRecord, track: str) -> Optional[str]:
    if track in log.df.columns:
        return track
    match = log.curve(track)
    return match.column if match else None


def plot_log_tracks(
    log: LogRecord,
    tracks: Sequence[str],
    title: Optional[str] = None,
    depth_column: Optional[str] = None,
    reverse_depth: bool = True,
    height: int = 760,
) -> go.Figure:
    depth_column = depth_column or physical_depth_column(log)
    resolved = [resolve_track(log, track) for track in tracks]
    resolved = [track for track in resolved if track is not None]
    if not resolved:
        raise ValueError('None of the requested tracks is available in this log.')

    fig = make_subplots(rows=1, cols=len(resolved), shared_yaxes=True, horizontal_spacing=0.035)
    for index, column in enumerate(resolved, start=1):
        temp = log.df[[depth_column, column]].apply(pd.to_numeric, errors='coerce').dropna()
        fig.add_trace(go.Scatter(
            x=temp[column], y=temp[depth_column], mode='lines',
            name=column, line={'width': 1.6, 'color': MS_COLOURS[(index - 1) % len(MS_COLOURS)]},
            hovertemplate=f'{column}: %{{x:.4g}}<br>Depth: %{{y:.3f}}<extra></extra>',
        ), row=1, col=index)
        unit = curve_unit(log, column)
        fig.update_xaxes(title_text=f'{column}<br><span style="font-size:10px">{unit}</span>', row=1, col=index)

    y_title = f'{depth_column} ({curve_unit(log, depth_column) or log.index_unit})'
    fig.update_yaxes(title_text=y_title, autorange='reversed' if reverse_depth else True, row=1, col=1)
    fig.update_layout(
        title=title or f'{log.log_id} — wireline tracks',
        template='plotly_white', height=height,
        width=max(900, 235 * len(resolved)),
        showlegend=False, hovermode='y unified',
        margin={'l': 70, 'r': 30, 't': 80, 'b': 60},
    )
    return fig


def find_curve_column(log: LogRecord, mnemonic: str, sonde_contains: Optional[str] = None) -> Optional[str]:
    curve = log.curve(mnemonic, sonde_contains=sonde_contains)
    return curve.column if curve else None

# Run 10 is the supplied ACS log with non-zero verticality channels and physically scaled T/C data.
directional_example_id = next((key for key in logs if 'Run10_10m_min_down' in key), None)
if directional_example_id:
    directional_example = logs[directional_example_id]
    example_tracks = [
        find_curve_column(directional_example, 'GR', 'Gamma'),
        find_curve_column(directional_example, 'Conductivity', 'Conductivity'),
        find_curve_column(directional_example, 'Temperature', 'Conductivity'),
        find_curve_column(directional_example, 'Incline', 'Verticality'),
        find_curve_column(directional_example, 'Azimuth', 'Verticality'),
    ]
    example_tracks = [track for track in example_tracks if track]
    plot_log_tracks(directional_example, example_tracks, title=f'{directional_example_id} — combined tool string').show()

## 8. Caliper calibration and calibrated/uncalibrated comparison

In [9]:
def polynomial_calibration(values: pd.Series, coefficients: Sequence[float]) -> pd.Series:
    c0, c1, c2, c3 = list(coefficients)[:4]
    numeric = pd.to_numeric(values, errors='coerce')
    return c0 + c1 * numeric + c2 * numeric.pow(2) + c3 * numeric.pow(3)


def caliper_looks_raw(log: LogRecord) -> bool:
    medians = []
    for arm in ARM_NAMES:
        curve = log.curve(arm)
        if curve:
            medians.append(float(log.df[curve.column].median()))
    return bool(medians) and float(np.nanmedian(medians)) > 10_000


def calibrated_caliper_frame(raw_log: LogRecord, calibration_source: LogRecord) -> pd.DataFrame:
    output = raw_log.df.copy()
    records = {
        record.channel_name: record
        for record in calibration_source.calibrations
        if 'caliper' in record.sonde_name.casefold()
        and record.channel_name in ARM_NAMES
        and record.mode != 'None'
    }
    missing = [arm for arm in ARM_NAMES if arm not in records]
    if missing:
        raise ValueError(f'Calibration source is missing records for: {missing}')
    for arm in ARM_NAMES:
        source_curve = raw_log.curve(arm)
        if source_curve is None:
            raise ValueError(f'Raw log is missing {arm}.')
        output[source_curve.column] = polynomial_calibration(output[source_curve.column], records[arm].coefficients)
    return output

print(f'{calibrated_id}: raw-count heuristic = {caliper_looks_raw(calibrated_log)}')
if uncalibrated_log is not None:
    print(f'{uncalibrated_id}: raw-count heuristic = {caliper_looks_raw(uncalibrated_log)}')
    uncalibrated_scaled = calibrated_caliper_frame(uncalibrated_log, calibrated_log)

    comparison_rows = []
    for arm in ARM_NAMES:
        raw_col = uncalibrated_log.curve(arm).column
        comparison_rows.append({
            'arm': arm,
            'raw_median': float(uncalibrated_log.df[raw_col].median()),
            'scaled_median_mm': float(uncalibrated_scaled[raw_col].median()),
            'calibrated_LAS_median_mm': float(calibrated_log.df[calibrated_log.curve(arm).column].median()),
        })
    display(pd.DataFrame(comparison_rows))


def compare_caliper_runs(
    physical_log: LogRecord,
    raw_log: LogRecord,
    raw_scaled_frame: pd.DataFrame,
    step_m: float = 0.02,
) -> pd.DataFrame:
    depth_a = physical_depth_column(physical_log)
    depth_b = physical_depth_column(raw_log)
    a = pd.DataFrame({'depth': physical_log.df[depth_a]})
    b = pd.DataFrame({'depth': raw_log.df[depth_b]})
    for arm in ARM_NAMES:
        a[arm] = physical_log.df[physical_log.curve(arm).column]
        b[arm] = raw_scaled_frame[raw_log.curve(arm).column]
    a = a.dropna().groupby('depth', as_index=False).median().sort_values('depth')
    b = b.dropna().groupby('depth', as_index=False).median().sort_values('depth')
    lower = max(a['depth'].min(), b['depth'].min())
    upper = min(a['depth'].max(), b['depth'].max())
    grid = np.arange(lower, upper + step_m / 2, step_m)
    rows = []
    for arm in ARM_NAMES:
        av = np.interp(grid, a['depth'], a[arm])
        bv = np.interp(grid, b['depth'], b[arm])
        rows.append({
            'arm': arm,
            'samples': len(grid),
            'correlation': float(np.corrcoef(av, bv)[0, 1]),
            'mean_bias_scaled_minus_calibrated_mm': float(np.mean(bv - av)),
            'MAE_mm': float(np.mean(np.abs(bv - av))),
            'RMSE_mm': float(np.sqrt(np.mean((bv - av) ** 2))),
        })
    return pd.DataFrame(rows)

if uncalibrated_log is not None:
    display(HTML('<div class="ms-callout"><b>Calibration cross-check.</b> These are separate logging runs, so residuals include repeatability, depth alignment and tool-position differences; they are not a pure calibration error.</div>'))
    display(compare_caliper_runs(calibrated_log, uncalibrated_log, uncalibrated_scaled))

Calibrated: raw-count heuristic = False
Uncalibrated: raw-count heuristic = True


,arm,raw_median,scaled_median_mm,calibrated_LAS_median_mm
0,X1,"3,780,178.0000",60.4652,61.4239
1,X2,"3,483,684.8000",68.3448,67.6801
2,Y1,"3,389,758.8000",71.4007,79.8488
3,Y2,"4,203,983.2000",79.5610,72.7597


,arm,samples,correlation,mean_bias_scaled_minus_calibrated_mm,MAE_mm,RMSE_mm
0,X1,1308,0.8969,-1.3956,1.4192,5.6410
1,X2,1308,0.8671,1.3264,1.7928,6.7415
2,Y1,1308,0.8990,-7.1809,7.2620,9.1767
3,Y2,1308,0.8889,4.6411,6.5997,8.9119


## 9. Four-arm borehole geometry, volume and reconciliation

In [10]:
def prepare_caliper_geometry(
    log: LogRecord,
    frame: Optional[pd.DataFrame] = None,
    mode: str = 'opposite_arm_sum',
    motor_current_min_ma: Optional[float] = 1.8,
    max_arm_mm: float = 1_000.0,
    nominal_diameter_mm: Optional[float] = None,
    max_gap_m: Optional[float] = None,
) -> pd.DataFrame:
    """
    Convert four independent arm measurements into orthogonal diameters and an elliptical area.

    mode='opposite_arm_sum': X diameter = X1+X2 and Y diameter = Y1+Y2.
    mode='per_arm_diameter_mean': X diameter = mean(X1,X2), Y diameter = mean(Y1,Y2).
    """
    frame = log.df if frame is None else frame
    depth_col = physical_depth_column(log)
    columns = {'depth_m': depth_col}
    for arm in ARM_NAMES:
        curve = log.curve(arm)
        if curve is None:
            raise ValueError(f'{log.log_id} is missing {arm}.')
        columns[arm] = curve.column

    data = pd.DataFrame({name: pd.to_numeric(frame[column], errors='coerce') for name, column in columns.items()})
    motor_curve = log.curve('MotorCurrent')
    if motor_curve and motor_curve.column in frame:
        data['motor_current_ma'] = pd.to_numeric(frame[motor_curve.column], errors='coerce')

    valid = data[list(ARM_NAMES) + ['depth_m']].notna().all(axis=1)
    for arm in ARM_NAMES:
        valid &= data[arm].between(0.0, max_arm_mm, inclusive='neither')
    if motor_current_min_ma is not None and 'motor_current_ma' in data:
        valid &= data['motor_current_ma'] >= motor_current_min_ma
    data = data.loc[valid].copy()
    if data.empty:
        raise ValueError('No valid caliper samples remain after QC filtering.')

    # Resolve repeated depth samples before integration.
    numeric_cols = [column for column in data.columns if column != 'depth_m']
    data = data.groupby('depth_m', as_index=False)[numeric_cols].median().sort_values('depth_m').reset_index(drop=True)

    if mode == 'opposite_arm_sum':
        data['diameter_x_mm'] = data['X1'] + data['X2']
        data['diameter_y_mm'] = data['Y1'] + data['Y2']
        data['hole_centre_x_mm'] = (data['X1'] - data['X2']) / 2.0
        data['hole_centre_y_mm'] = (data['Y1'] - data['Y2']) / 2.0
    elif mode == 'per_arm_diameter_mean':
        data['diameter_x_mm'] = (data['X1'] + data['X2']) / 2.0
        data['diameter_y_mm'] = (data['Y1'] + data['Y2']) / 2.0
        data['hole_centre_x_mm'] = np.nan
        data['hole_centre_y_mm'] = np.nan
    else:
        raise ValueError("mode must be 'opposite_arm_sum' or 'per_arm_diameter_mean'.")

    data['semi_axis_x_mm'] = data['diameter_x_mm'] / 2.0
    data['semi_axis_y_mm'] = data['diameter_y_mm'] / 2.0
    data['major_diameter_mm'] = data[['diameter_x_mm', 'diameter_y_mm']].max(axis=1)
    data['minor_diameter_mm'] = data[['diameter_x_mm', 'diameter_y_mm']].min(axis=1)
    data['area_mm2'] = math.pi * data['semi_axis_x_mm'] * data['semi_axis_y_mm']
    data['area_m2'] = data['area_mm2'] / 1_000_000.0
    data['equivalent_diameter_mm'] = np.sqrt(data['diameter_x_mm'] * data['diameter_y_mm'])
    mean_diameter = (data['major_diameter_mm'] + data['minor_diameter_mm']) / 2.0
    data['ovality_pct'] = 100.0 * (data['major_diameter_mm'] - data['minor_diameter_mm']) / mean_diameter
    data['tool_offset_mm'] = np.hypot(data['hole_centre_x_mm'], data['hole_centre_y_mm'])

    data['delta_depth_m'] = data['depth_m'].diff()
    positive_steps = data.loc[data['delta_depth_m'] > 0, 'delta_depth_m']
    median_step = float(positive_steps.median()) if not positive_steps.empty else np.nan
    if max_gap_m is None:
        max_gap_m = max(0.10, 5.0 * median_step) if np.isfinite(median_step) else 0.10
    segment_valid = data['delta_depth_m'].between(0.0, max_gap_m, inclusive='right')
    segment_area = (data['area_m2'] + data['area_m2'].shift(1)) / 2.0
    data['segment_volume_m3'] = (segment_area * data['delta_depth_m']).where(segment_valid, 0.0)
    data['cumulative_volume_m3'] = data['segment_volume_m3'].cumsum()

    if nominal_diameter_mm is not None:
        nominal_area = math.pi * (nominal_diameter_mm / 2_000.0) ** 2
        data['nominal_area_m2'] = nominal_area
        data['segment_nominal_volume_m3'] = (nominal_area * data['delta_depth_m']).where(segment_valid, 0.0)
        data['cumulative_nominal_volume_m3'] = data['segment_nominal_volume_m3'].cumsum()
        data['segment_volume_variance_m3'] = data['segment_volume_m3'] - data['segment_nominal_volume_m3']
        data['cumulative_volume_variance_m3'] = data['segment_volume_variance_m3'].cumsum()

    data.attrs.update({
        'log_id': log.log_id,
        'geometry_mode': mode,
        'median_step_m': median_step,
        'max_gap_m': max_gap_m,
        'nominal_diameter_mm': nominal_diameter_mm,
    })
    return data


def caliper_reconciliation_summary(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
) -> pd.DataFrame:
    segment_length = geometry['delta_depth_m'].where(geometry['segment_volume_m3'] > 0, 0.0).fillna(0.0)
    segment_diameter = (geometry['equivalent_diameter_mm'] + geometry['equivalent_diameter_mm'].shift(1)) / 2.0
    total_length = float(segment_length.sum())
    within = (segment_diameter - nominal_diameter_mm).abs() <= tolerance_mm
    below = segment_diameter < nominal_diameter_mm - tolerance_mm
    above = segment_diameter > nominal_diameter_mm + tolerance_mm

    measured_volume = float(geometry['segment_volume_m3'].sum())
    nominal_volume = float(geometry.get('segment_nominal_volume_m3', pd.Series(0.0, index=geometry.index)).sum())
    variance = measured_volume - nominal_volume
    within_pct = 100.0 * float(segment_length.where(within, 0.0).sum()) / total_length if total_length else np.nan

    return pd.DataFrame([{
        'log_id': geometry.attrs.get('log_id'),
        'geometry_mode': geometry.attrs.get('geometry_mode'),
        'valid_depth_from_m': float(geometry['depth_m'].min()),
        'valid_depth_to_m': float(geometry['depth_m'].max()),
        'integrated_length_m': total_length,
        'equivalent_diameter_p10_mm': float(geometry['equivalent_diameter_mm'].quantile(0.10)),
        'equivalent_diameter_median_mm': float(geometry['equivalent_diameter_mm'].median()),
        'equivalent_diameter_p90_mm': float(geometry['equivalent_diameter_mm'].quantile(0.90)),
        'mean_ovality_pct': float(geometry['ovality_pct'].mean()),
        'median_tool_offset_mm': float(geometry['tool_offset_mm'].median()) if geometry['tool_offset_mm'].notna().any() else np.nan,
        'nominal_diameter_mm': nominal_diameter_mm,
        'tolerance_mm': tolerance_mm,
        'length_within_tolerance_pct': within_pct,
        'length_below_tolerance_m': float(segment_length.where(below, 0.0).sum()),
        'length_above_tolerance_m': float(segment_length.where(above, 0.0).sum()),
        'measured_volume_m3': measured_volume,
        'nominal_volume_m3': nominal_volume,
        'volume_variance_m3': variance,
        'volume_variance_pct': 100.0 * variance / nominal_volume if nominal_volume else np.nan,
        'diameter_status': 'PASS' if within_pct >= 95.0 else 'REVIEW',
    }])

caliper_geometry = prepare_caliper_geometry(
    calibrated_log,
    mode=CALIPER_GEOMETRY_MODE,
    motor_current_min_ma=CALIPER_MOTOR_CURRENT_MIN_MA,
    max_arm_mm=CALIPER_MAX_ARM_MM,
    nominal_diameter_mm=NOMINAL_DIAMETER_MM,
)
caliper_summary = caliper_reconciliation_summary(caliper_geometry, NOMINAL_DIAMETER_MM, DIAMETER_TOLERANCE_MM)
display(caliper_summary)

# Show the consequence of the unresolved channel convention rather than hiding it.
mode_comparison = []
for geometry_mode in ['opposite_arm_sum', 'per_arm_diameter_mean']:
    candidate = prepare_caliper_geometry(
        calibrated_log,
        mode=geometry_mode,
        motor_current_min_ma=CALIPER_MOTOR_CURRENT_MIN_MA,
        max_arm_mm=CALIPER_MAX_ARM_MM,
        nominal_diameter_mm=NOMINAL_DIAMETER_MM,
    )
    mode_comparison.append(caliper_reconciliation_summary(candidate, NOMINAL_DIAMETER_MM, DIAMETER_TOLERANCE_MM))
display(HTML('<div class="ms-warning"><b>Geometry convention sensitivity.</b> Production calculations must use the convention confirmed for this sonde and calibration workflow.</div>'))
display(pd.concat(mode_comparison, ignore_index=True))

,log_id,geometry_mode,valid_depth_from_m,valid_depth_to_m,integrated_length_m,equivalent_diameter_p10_mm,equivalent_diameter_median_mm,equivalent_diameter_p90_mm,mean_ovality_pct,median_tool_offset_mm,nominal_diameter_mm,tolerance_mm,length_within_tolerance_pct,length_below_tolerance_m,length_above_tolerance_m,measured_volume_m3,nominal_volume_m3,volume_variance_m3,volume_variance_pct,diameter_status
0,Calibrated,opposite_arm_sum,2.0200,27.9500,25.9300,89.5404,140.3796,140.8135,20.4492,4.9230,150.0000,10.0000,79.3290,5.3600,0.0000,0.3533,0.4582,-0.1049,-22.8980,REVIEW


,log_id,geometry_mode,valid_depth_from_m,valid_depth_to_m,integrated_length_m,equivalent_diameter_p10_mm,equivalent_diameter_median_mm,equivalent_diameter_p90_mm,mean_ovality_pct,median_tool_offset_mm,nominal_diameter_mm,tolerance_mm,length_within_tolerance_pct,length_below_tolerance_m,length_above_tolerance_m,measured_volume_m3,nominal_volume_m3,volume_variance_m3,volume_variance_pct,diameter_status
0,Calibrated,opposite_arm_sum,2.0200,27.9500,25.9300,89.5404,140.3796,140.8135,20.4492,4.9230,150.0000,10.0000,79.3290,5.3600,0.0000,0.3533,0.4582,-0.1049,-22.8980,REVIEW
1,Calibrated,per_arm_diameter_mean,2.0200,27.9500,25.9300,44.7702,70.1898,70.4067,20.4492,NaN,150.0000,10.0000,0.0000,25.9300,0.0000,0.0883,0.4582,-0.3699,-80.7245,REVIEW


In [11]:
def metric_cards(summary: pd.Series) -> HTML:
    values = [
        ('Integrated length', f"{summary['integrated_length_m']:.2f} m"),
        ('Median equivalent diameter', f"{summary['equivalent_diameter_median_mm']:.1f} mm"),
        ('Measured volume', f"{summary['measured_volume_m3']:.3f} m³"),
        ('Volume variance', f"{summary['volume_variance_pct']:+.1f}%"),
        ('Within tolerance', f"{summary['length_within_tolerance_pct']:.1f}%"),
        ('Status', str(summary['diameter_status'])),
    ]
    cards = ''.join(f'<div class="ms-card"><div class="label">{label}</div><div class="value">{value}</div></div>' for label, value in values)
    return HTML(cards)

display(metric_cards(caliper_summary.iloc[0]))


def caliper_interval_summary(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
    interval_m: float = 1.0,
) -> pd.DataFrame:
    output = geometry.copy()
    origin = math.floor(output['depth_m'].min() / interval_m) * interval_m
    output['interval_from_m'] = origin + np.floor((output['depth_m'] - origin) / interval_m) * interval_m
    output['interval_to_m'] = output['interval_from_m'] + interval_m
    grouped = output.groupby(['interval_from_m', 'interval_to_m'], as_index=False).agg(
        samples=('depth_m', 'size'),
        equivalent_diameter_mean_mm=('equivalent_diameter_mm', 'mean'),
        equivalent_diameter_min_mm=('equivalent_diameter_mm', 'min'),
        equivalent_diameter_max_mm=('equivalent_diameter_mm', 'max'),
        ovality_max_pct=('ovality_pct', 'max'),
        measured_volume_m3=('segment_volume_m3', 'sum'),
        nominal_volume_m3=('segment_nominal_volume_m3', 'sum'),
    )
    grouped['volume_variance_m3'] = grouped['measured_volume_m3'] - grouped['nominal_volume_m3']
    grouped['diameter_variance_mm'] = grouped['equivalent_diameter_mean_mm'] - nominal_diameter_mm
    grouped['status'] = np.where(grouped['diameter_variance_mm'].abs() <= tolerance_mm, 'PASS', 'REVIEW')
    return grouped

interval_summary = caliper_interval_summary(caliper_geometry, NOMINAL_DIAMETER_MM, DIAMETER_TOLERANCE_MM)
display(interval_summary)

,interval_from_m,interval_to_m,samples,equivalent_diameter_mean_mm,equivalent_diameter_min_mm,equivalent_diameter_max_mm,ovality_max_pct,measured_volume_m3,nominal_volume_m3,volume_variance_m3,diameter_variance_mm,status
0,2.0000,3.0000,98,140.2217,140.2001,140.2488,16.5763,0.0150,0.0171,-0.0022,-9.7783,PASS
1,3.0000,4.0000,100,140.2199,140.2036,140.2506,16.5757,0.0154,0.0177,-0.0022,-9.7801,PASS
2,4.0000,5.0000,100,140.2729,140.2080,140.3597,16.7257,0.0155,0.0177,-0.0022,-9.7271,PASS
3,5.0000,6.0000,100,140.3411,140.3199,140.3705,16.7249,0.0155,0.0177,-0.0022,-9.6589,PASS
4,6.0000,7.0000,100,140.3436,140.3240,140.3737,16.7238,0.0155,0.0177,-0.0022,-9.6564,PASS
5,7.0000,8.0000,100,140.3580,140.3299,140.3827,16.7176,0.0155,0.0177,-0.0022,-9.6420,PASS
6,8.0000,9.0000,100,140.3617,140.3387,140.3877,16.7097,0.0155,0.0177,-0.0022,-9.6383,PASS
7,9.0000,10.0000,100,140.3662,140.3427,140.3915,16.7041,0.0155,0.0177,-0.0022,-9.6338,PASS
8,10.0000,11.0000,100,140.4715,140.3470,140.6230,17.0221,0.0155,0.0177,-0.0022,-9.5285,PASS
9,11.0000,12.0000,100,140.6031,140.5779,140.6275,17.0254,0.0155,0.0177,-0.0021,-9.3969,PASS


## 10. Caliper visualisations

In [12]:
def plot_caliper_dashboard(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=5, shared_yaxes=True, horizontal_spacing=0.035,
                        subplot_titles=('Independent arms', 'Opposite-pair diameters', 'Equivalent diameter', 'Ovality / offset', 'Cumulative volume'))
    depth = geometry['depth_m']

    for index, arm in enumerate(ARM_NAMES):
        fig.add_trace(go.Scatter(x=geometry[arm], y=depth, mode='lines', name=arm,
                                 line={'width': 1.2, 'color': MS_COLOURS[index]}), row=1, col=1)
    for index, column in enumerate(['diameter_x_mm', 'diameter_y_mm']):
        fig.add_trace(go.Scatter(x=geometry[column], y=depth, mode='lines', name=column,
                                 line={'width': 1.5, 'color': MS_COLOURS[index + 1]}), row=1, col=2)
    for value, dash, name in [
        (nominal_diameter_mm, 'solid', 'Nominal'),
        (nominal_diameter_mm - tolerance_mm, 'dot', 'Lower tolerance'),
        (nominal_diameter_mm + tolerance_mm, 'dot', 'Upper tolerance'),
    ]:
        fig.add_trace(go.Scatter(x=np.full(len(depth), value), y=depth, mode='lines', name=name,
                                 line={'width': 1.0, 'dash': dash, 'color': '#59636E'}), row=1, col=2)

    fig.add_trace(go.Scatter(x=geometry['equivalent_diameter_mm'], y=depth, mode='lines', name='Equivalent diameter',
                             line={'width': 1.8, 'color': MS_COLOURS[2]}), row=1, col=3)
    for value, dash, name in [
        (nominal_diameter_mm, 'solid', 'Nominal equivalent'),
        (nominal_diameter_mm - tolerance_mm, 'dot', 'Lower tolerance'),
        (nominal_diameter_mm + tolerance_mm, 'dot', 'Upper tolerance'),
    ]:
        fig.add_trace(go.Scatter(x=np.full(len(depth), value), y=depth, mode='lines', name=name,
                                 line={'width': 1.0, 'dash': dash, 'color': '#59636E'}), row=1, col=3)

    fig.add_trace(go.Scatter(x=geometry['ovality_pct'], y=depth, mode='lines', name='Ovality %',
                             line={'width': 1.5, 'color': MS_COLOURS[5]}), row=1, col=4)
    if geometry['tool_offset_mm'].notna().any():
        fig.add_trace(go.Scatter(x=geometry['tool_offset_mm'], y=depth, mode='lines', name='Tool offset mm',
                                 line={'width': 1.2, 'color': MS_COLOURS[6], 'dash': 'dash'}), row=1, col=4)

    fig.add_trace(go.Scatter(x=geometry['cumulative_volume_m3'], y=depth, mode='lines', name='Measured volume',
                             line={'width': 1.8, 'color': MS_COLOURS[0]}), row=1, col=5)
    fig.add_trace(go.Scatter(x=geometry['cumulative_nominal_volume_m3'], y=depth, mode='lines', name='Nominal volume',
                             line={'width': 1.3, 'color': MS_COLOURS[4], 'dash': 'dash'}), row=1, col=5)

    fig.update_yaxes(title_text='Measured depth (m)', autorange='reversed', row=1, col=1)
    fig.update_xaxes(title_text='Arm distance (mm)', row=1, col=1)
    fig.update_xaxes(title_text='Diameter (mm)', row=1, col=2)
    fig.update_xaxes(title_text='Diameter (mm)', row=1, col=3)
    fig.update_xaxes(title_text='% or mm', row=1, col=4)
    fig.update_xaxes(title_text='Volume (m³)', row=1, col=5)
    fig.update_layout(
        title=f"{geometry.attrs.get('log_id')} — caliper geometry and volume ({geometry.attrs.get('geometry_mode')})",
        template='plotly_white', height=820, width=1500, hovermode='y unified',
        legend={'orientation': 'h', 'y': -0.12}, margin={'l': 70, 'r': 30, 't': 95, 'b': 115},
    )
    return fig

plot_caliper_dashboard(caliper_geometry, NOMINAL_DIAMETER_MM, DIAMETER_TOLERANCE_MM).show()

In [13]:
def nearest_caliper_row(geometry: pd.DataFrame, target_depth_m: float) -> pd.Series:
    index = (geometry['depth_m'] - target_depth_m).abs().idxmin()
    return geometry.loc[index]


def plot_caliper_cross_section(
    geometry: pd.DataFrame,
    target_depth_m: float,
    nominal_diameter_mm: Optional[float] = None,
) -> go.Figure:
    row = nearest_caliper_row(geometry, target_depth_m)
    theta = np.linspace(0, 2 * math.pi, 181)
    cx = float(row['hole_centre_x_mm']) if pd.notna(row['hole_centre_x_mm']) else 0.0
    cy = float(row['hole_centre_y_mm']) if pd.notna(row['hole_centre_y_mm']) else 0.0
    sx = float(row['semi_axis_x_mm'])
    sy = float(row['semi_axis_y_mm'])
    ellipse_x = cx + sx * np.cos(theta)
    ellipse_y = cy + sy * np.sin(theta)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ellipse_x, y=ellipse_y, mode='lines', name='Inferred elliptical wall',
                             line={'width': 3, 'color': MS_COLOURS[1]}))
    if nominal_diameter_mm is not None:
        radius = nominal_diameter_mm / 2.0
        fig.add_trace(go.Scatter(x=cx + radius * np.cos(theta), y=cy + radius * np.sin(theta), mode='lines',
                                 name='Nominal circle', line={'width': 1.5, 'dash': 'dash', 'color': MS_COLOURS[4]}))

    contact_x = [float(row['X1']), -float(row['X2']), 0.0, 0.0]
    contact_y = [0.0, 0.0, float(row['Y1']), -float(row['Y2'])]
    labels = ['X1', 'X2', 'Y1', 'Y2']
    for x, y, label in zip(contact_x, contact_y, labels):
        fig.add_trace(go.Scatter(x=[0, x], y=[0, y], mode='lines+markers+text', text=['', label], textposition='top center',
                                 name=label, line={'width': 1.2}, marker={'size': [6, 9]}))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers', name='Tool centre', marker={'size': 11, 'symbol': 'x', 'color': '#111111'}))
    fig.add_trace(go.Scatter(x=[cx], y=[cy], mode='markers', name='Inferred hole centre', marker={'size': 10, 'symbol': 'diamond', 'color': MS_COLOURS[5]}))

    fig.update_xaxes(title='X (mm)', scaleanchor='y', scaleratio=1, zeroline=True)
    fig.update_yaxes(title='Y (mm)', zeroline=True)
    fig.update_layout(
        title=(f"Caliper cross-section at {row['depth_m']:.3f} m — "
               f"Dx={row['diameter_x_mm']:.1f} mm, Dy={row['diameter_y_mm']:.1f} mm, "
               f"Deq={row['equivalent_diameter_mm']:.1f} mm"),
        template='plotly_white', width=800, height=720,
        legend={'orientation': 'h', 'y': -0.12}, margin={'l': 70, 'r': 30, 't': 85, 'b': 110},
    )
    return fig

CROSS_SECTION_DEPTH_M = float(caliper_geometry['depth_m'].median())
plot_caliper_cross_section(caliper_geometry, CROSS_SECTION_DEPTH_M, NOMINAL_DIAMETER_MM).show()

In [14]:
def plot_caliper_tube(geometry: pd.DataFrame, max_rings: int = 100, angular_samples: int = 40) -> go.Figure:
    if geometry['hole_centre_x_mm'].isna().all():
        raise ValueError('A 3D centred tube requires the opposite-arm-sum geometry mode.')
    stride = max(1, math.ceil(len(geometry) / max_rings))
    sample = geometry.iloc[::stride].copy()
    theta = np.linspace(0, 2 * math.pi, angular_samples)
    x = (sample['hole_centre_x_mm'].to_numpy()[:, None] + sample['semi_axis_x_mm'].to_numpy()[:, None] * np.cos(theta)[None, :]) / 1000.0
    y = (sample['hole_centre_y_mm'].to_numpy()[:, None] + sample['semi_axis_y_mm'].to_numpy()[:, None] * np.sin(theta)[None, :]) / 1000.0
    z = np.repeat(sample['depth_m'].to_numpy()[:, None], angular_samples, axis=1)

    fig = go.Figure()
    fig.add_trace(go.Surface(x=x, y=y, z=z, surfacecolor=z, colorscale='Viridis', opacity=0.82,
                             colorbar={'title': 'Depth (m)'}, showscale=True, name='Borehole wall'))
    fig.add_trace(go.Scatter3d(x=sample['hole_centre_x_mm'] / 1000.0, y=sample['hole_centre_y_mm'] / 1000.0,
                               z=sample['depth_m'], mode='lines', name='Inferred hole centreline',
                               line={'width': 5, 'color': '#111111'}))
    fig.update_layout(
        title='3D caliper-derived borehole envelope', template='plotly_white', width=900, height=760,
        scene={
            'xaxis_title': 'X (m)', 'yaxis_title': 'Y (m)', 'zaxis_title': 'Measured depth (m)',
            'zaxis': {'autorange': 'reversed'}, 'aspectmode': 'manual',
            'aspectratio': {'x': 1, 'y': 1, 'z': 4},
        },
        margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
    )
    return fig

plot_caliper_tube(caliper_geometry).show()

## 11. Hole plan / inspection register and reconciliation pattern

In [15]:
# Replace this example with a CSV or database extract from the drill-and-blast plan.
hole_register = pd.DataFrame([
    {
        'hole_id': calibrated_id,
        'planned_depth_m': 28.0,
        'nominal_diameter_mm': NOMINAL_DIAMETER_MM,
        'diameter_tolerance_mm': DIAMETER_TOLERANCE_MM,
        'planned_inclination_from_vertical_deg': 0.0,
        'planned_azimuth_deg': 0.0,
        'collar_easting_m': np.nan,
        'collar_northing_m': np.nan,
        'collar_elevation_m': np.nan,
    }
])
display(hole_register)


def reconcile_caliper_to_register(log: LogRecord, register_row: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    geometry = prepare_caliper_geometry(
        log,
        mode=CALIPER_GEOMETRY_MODE,
        motor_current_min_ma=CALIPER_MOTOR_CURRENT_MIN_MA,
        max_arm_mm=CALIPER_MAX_ARM_MM,
        nominal_diameter_mm=float(register_row['nominal_diameter_mm']),
    )
    summary = caliper_reconciliation_summary(
        geometry,
        nominal_diameter_mm=float(register_row['nominal_diameter_mm']),
        tolerance_mm=float(register_row['diameter_tolerance_mm']),
    )
    summary['planned_depth_m'] = float(register_row['planned_depth_m'])
    summary['logged_to_depth_m'] = float(geometry['depth_m'].max())
    summary['depth_variance_m'] = summary['logged_to_depth_m'] - summary['planned_depth_m']
    return geometry, summary

register_geometry, register_reconciliation = reconcile_caliper_to_register(
    calibrated_log,
    hole_register.loc[hole_register['hole_id'].eq(calibrated_id)].iloc[0],
)
display(register_reconciliation)

,hole_id,planned_depth_m,nominal_diameter_mm,diameter_tolerance_mm,planned_inclination_from_vertical_deg,planned_azimuth_deg,collar_easting_m,collar_northing_m,collar_elevation_m
0,Calibrated,28.0000,150.0000,10.0000,0.0000,0.0000,NaN,NaN,NaN


,log_id,geometry_mode,valid_depth_from_m,valid_depth_to_m,integrated_length_m,equivalent_diameter_p10_mm,equivalent_diameter_median_mm,equivalent_diameter_p90_mm,mean_ovality_pct,median_tool_offset_mm,nominal_diameter_mm,tolerance_mm,length_within_tolerance_pct,length_below_tolerance_m,length_above_tolerance_m,measured_volume_m3,nominal_volume_m3,volume_variance_m3,volume_variance_pct,diameter_status,planned_depth_m,logged_to_depth_m,depth_variance_m
0,Calibrated,opposite_arm_sum,2.0200,27.9500,25.9300,89.5404,140.3796,140.8135,20.4492,4.9230,150.0000,10.0000,79.3290,5.3600,0.0000,0.3533,0.4582,-0.1049,-22.8980,REVIEW,28.0000,27.9500,-0.0500


## 12. Directional survey and 3D borehole trajectory

In [16]:
def compute_minimum_curvature_trajectory(
    log: LogRecord,
    inclination_reference: str = 'vertical',
) -> pd.DataFrame:
    depth_col = physical_depth_column(log)
    inclination_col = find_curve_column(log, 'Incline', 'Verticality') or find_curve_column(log, 'Incline')
    azimuth_col = find_curve_column(log, 'Azimuth', 'Verticality') or find_curve_column(log, 'Azimuth')
    if not inclination_col or not azimuth_col:
        raise ValueError(f'{log.log_id} does not contain inclination and azimuth curves.')

    survey = pd.DataFrame({
        'measured_depth_m': pd.to_numeric(log.df[depth_col], errors='coerce'),
        'inclination_deg': pd.to_numeric(log.df[inclination_col], errors='coerce'),
        'azimuth_deg': pd.to_numeric(log.df[azimuth_col], errors='coerce'),
    }).dropna()
    survey = survey.groupby('measured_depth_m', as_index=False).median().sort_values('measured_depth_m').reset_index(drop=True)
    survey = survey.loc[survey['measured_depth_m'].diff().fillna(1.0) > 0].reset_index(drop=True)
    if len(survey) < 2:
        raise ValueError('At least two valid directional stations are required.')

    inclination = survey['inclination_deg'].to_numpy(dtype=float)
    if inclination_reference.casefold() == 'horizontal':
        inclination = 90.0 - inclination
    elif inclination_reference.casefold() != 'vertical':
        raise ValueError("inclination_reference must be 'vertical' or 'horizontal'.")

    inc = np.deg2rad(inclination)
    azi = np.deg2rad(np.mod(survey['azimuth_deg'].to_numpy(dtype=float), 360.0))
    md = survey['measured_depth_m'].to_numpy(dtype=float)
    dmd = np.diff(md)

    inc1, inc2 = inc[:-1], inc[1:]
    azi1, azi2 = azi[:-1], azi[1:]
    cosine_dogleg = np.cos(inc1) * np.cos(inc2) + np.sin(inc1) * np.sin(inc2) * np.cos(azi2 - azi1)
    dogleg = np.arccos(np.clip(cosine_dogleg, -1.0, 1.0))
    ratio_factor = np.ones_like(dogleg)
    nonzero = dogleg > 1e-10
    ratio_factor[nonzero] = 2.0 * np.tan(dogleg[nonzero] / 2.0) / dogleg[nonzero]

    d_north = 0.5 * dmd * (np.sin(inc1) * np.cos(azi1) + np.sin(inc2) * np.cos(azi2)) * ratio_factor
    d_east = 0.5 * dmd * (np.sin(inc1) * np.sin(azi1) + np.sin(inc2) * np.sin(azi2)) * ratio_factor
    d_tvd = 0.5 * dmd * (np.cos(inc1) + np.cos(inc2)) * ratio_factor

    survey['inclination_from_vertical_deg'] = inclination
    survey['northing_offset_m'] = np.r_[0.0, np.cumsum(d_north)]
    survey['easting_offset_m'] = np.r_[0.0, np.cumsum(d_east)]
    survey['tvd_m'] = np.r_[0.0, np.cumsum(d_tvd)]
    survey['horizontal_departure_m'] = np.hypot(survey['easting_offset_m'], survey['northing_offset_m'])
    survey['dogleg_deg'] = np.r_[np.nan, np.rad2deg(dogleg)]
    survey['dogleg_severity_deg_per_30m'] = np.r_[np.nan, np.rad2deg(dogleg) / dmd * 30.0]
    survey.attrs['log_id'] = log.log_id
    return survey


def trajectory_summary(trajectory: pd.DataFrame) -> pd.DataFrame:
    toe = trajectory.iloc[-1]
    return pd.DataFrame([{
        'log_id': trajectory.attrs.get('log_id'),
        'survey_from_md_m': float(trajectory['measured_depth_m'].min()),
        'survey_to_md_m': float(trajectory['measured_depth_m'].max()),
        'survey_interval_m': float(trajectory['measured_depth_m'].max() - trajectory['measured_depth_m'].min()),
        'toe_tvd_m': float(toe['tvd_m']),
        'toe_easting_offset_m': float(toe['easting_offset_m']),
        'toe_northing_offset_m': float(toe['northing_offset_m']),
        'toe_horizontal_departure_m': float(toe['horizontal_departure_m']),
        'maximum_dogleg_severity_deg_per_30m': float(trajectory['dogleg_severity_deg_per_30m'].max()),
    }])


def plot_trajectory(trajectory: pd.DataFrame) -> go.Figure:
    fig = go.Figure(go.Scatter3d(
        x=trajectory['easting_offset_m'], y=trajectory['northing_offset_m'], z=trajectory['tvd_m'],
        mode='lines+markers', name='Measured trajectory',
        line={'width': 6, 'color': MS_COLOURS[1]}, marker={'size': 2},
        customdata=np.column_stack([
            trajectory['measured_depth_m'], trajectory['inclination_from_vertical_deg'], trajectory['azimuth_deg']
        ]),
        hovertemplate='E: %{x:.3f} m<br>N: %{y:.3f} m<br>TVD: %{z:.3f} m<br>MD: %{customdata[0]:.3f} m<br>Inc: %{customdata[1]:.2f}°<br>Azi: %{customdata[2]:.2f}°<extra></extra>',
    ))
    fig.update_layout(
        title=f"{trajectory.attrs.get('log_id')} — minimum-curvature trajectory",
        template='plotly_white', width=900, height=760,
        scene={
            'xaxis_title': 'Easting offset (m)', 'yaxis_title': 'Northing offset (m)', 'zaxis_title': 'TVD down (m)',
            'zaxis': {'autorange': 'reversed'}, 'aspectmode': 'data',
        }, margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
    )
    return fig

trajectory = None
if directional_example_id:
    trajectory = compute_minimum_curvature_trajectory(directional_example, TRAJECTORY_INCLINATION_REFERENCE)
    display(trajectory_summary(trajectory))
    plot_trajectory(trajectory).show()

,log_id,survey_from_md_m,survey_to_md_m,survey_interval_m,toe_tvd_m,toe_easting_offset_m,toe_northing_offset_m,toe_horizontal_departure_m,maximum_dogleg_severity_deg_per_30m
0,ACS-03_Run10_10m_min_down,2.0500,20.4100,18.3600,18.3598,-0.0174,0.0367,0.0406,441.3835


The trajectory calculation assumes inclination is measured from vertical and azimuth is clockwise from north. Those are common survey conventions, but the app should store the convention explicitly with every data source and never infer it silently.

> **Near-vertical survey caution:** the selected ACS run has a median inclination of about 0.2°. At very low inclination, azimuth is poorly constrained and small station-to-station changes can create a very large normalised dogleg-severity value even though the computed toe departure is only about 0.04 m. For these holes, review inclination, total departure and instrument QC together; do not use the maximum DLS in isolation.

## 13. Repeated-run comparison for calibration and field repeatability

In [17]:
def interpolate_track(log: LogRecord, column: str, grid: np.ndarray) -> np.ndarray:
    depth_col = physical_depth_column(log)
    data = pd.DataFrame({
        'depth': pd.to_numeric(log.df[depth_col], errors='coerce'),
        'value': pd.to_numeric(log.df[column], errors='coerce'),
    }).dropna()
    data = data.groupby('depth', as_index=False).median().sort_values('depth')
    return np.interp(grid, data['depth'], data['value'])


def repeated_run_matrix(
    selected_logs: Sequence[LogRecord],
    mnemonic: str,
    grid_step_m: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prepared = []
    for log in selected_logs:
        column = find_curve_column(log, mnemonic)
        if not column:
            continue
        depth = pd.to_numeric(log.df[physical_depth_column(log)], errors='coerce').dropna()
        values = pd.to_numeric(log.df[column], errors='coerce').dropna()
        if len(depth) < 2 or len(values) < 2:
            continue
        prepared.append((log, column, float(depth.min()), float(depth.max())))
    if len(prepared) < 2:
        raise ValueError('At least two comparable runs are required.')

    lower = max(item[2] for item in prepared)
    upper = min(item[3] for item in prepared)
    grid = np.arange(lower, upper + grid_step_m / 2.0, grid_step_m)
    matrix = pd.DataFrame({'depth_m': grid})
    for log, column, _, _ in prepared:
        matrix[log.log_id] = interpolate_track(log, column, grid)
    value_columns = [column for column in matrix.columns if column != 'depth_m']
    matrix['ensemble_mean'] = matrix[value_columns].mean(axis=1)
    matrix['ensemble_std'] = matrix[value_columns].std(axis=1)

    metrics = []
    for column in value_columns:
        residual = matrix[column] - matrix['ensemble_mean']
        metrics.append({
            'log_id': column,
            'bias_to_ensemble': float(residual.mean()),
            'MAE_to_ensemble': float(residual.abs().mean()),
            'RMSE_to_ensemble': float(np.sqrt(np.mean(residual ** 2))),
            'correlation_to_ensemble': float(matrix[[column, 'ensemble_mean']].corr().iloc[0, 1]),
        })
    return matrix, pd.DataFrame(metrics)


def plot_repeated_runs(matrix: pd.DataFrame, mnemonic: str, unit: str = '') -> go.Figure:
    value_columns = [column for column in matrix.columns if column not in {'depth_m', 'ensemble_mean', 'ensemble_std'}]
    fig = go.Figure()
    for index, column in enumerate(value_columns):
        fig.add_trace(go.Scatter(x=matrix[column], y=matrix['depth_m'], mode='lines', name=column,
                                 opacity=0.48, line={'width': 1.0, 'color': MS_COLOURS[index % len(MS_COLOURS)]}))
    upper = matrix['ensemble_mean'] + 2.0 * matrix['ensemble_std']
    lower = matrix['ensemble_mean'] - 2.0 * matrix['ensemble_std']
    fig.add_trace(go.Scatter(x=upper, y=matrix['depth_m'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=lower, y=matrix['depth_m'], mode='lines', fill='tonextx', name='Ensemble ±2σ',
                             line={'width': 0}, fillcolor='rgba(42,157,143,0.18)', hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=matrix['ensemble_mean'], y=matrix['depth_m'], mode='lines', name='Ensemble mean',
                             line={'width': 3, 'color': '#111111'}))
    fig.update_yaxes(title='Measured depth (m)', autorange='reversed')
    fig.update_xaxes(title=f'{mnemonic} {f"({unit})" if unit else ""}')
    fig.update_layout(title=f'Repeated-run comparison — {mnemonic}', template='plotly_white', width=900, height=760,
                      legend={'orientation': 'h', 'y': -0.14}, margin={'l': 70, 'r': 30, 't': 80, 'b': 125})
    return fig

# Use the first seven upward ACS runs: they overlap over a useful interval and contain gamma data.
repeat_logs = [
    log for key, log in logs.items()
    if re.fullmatch(r'ACS-03_Run[1-7]', key) and (log.view_type or '').casefold() == 'up'
]
if len(repeat_logs) >= 2:
    gamma_matrix, gamma_repeatability = repeated_run_matrix(repeat_logs, 'GR', grid_step_m=0.05)
    display(gamma_repeatability)
    gamma_unit = curve_unit(repeat_logs[0], find_curve_column(repeat_logs[0], 'GR'))
    plot_repeated_runs(gamma_matrix, 'GR', gamma_unit).show()

,log_id,bias_to_ensemble,MAE_to_ensemble,RMSE_to_ensemble,correlation_to_ensemble
0,ACS-03_Run1,0.7104,4.2493,5.6260,0.8346
1,ACS-03_Run2,-0.2623,4.2983,5.5039,0.8590
2,ACS-03_Run3,-0.5340,4.3284,5.6399,0.8289
3,ACS-03_Run4,0.2008,4.0290,5.5114,0.8652
4,ACS-03_Run5,0.1248,4.0524,5.2658,0.8283
5,ACS-03_Run6,0.3309,3.8876,5.0858,0.8245
6,ACS-03_Run7,-0.5707,3.9887,5.0289,0.8311


## 14. Export-ready data products for the MiningSim application

In [18]:
def build_log_metadata_payload(log: LogRecord) -> dict[str, Any]:
    return {
        'hole_id': log.log_id,
        'source_archive': log.archive_path.name,
        'source_las_member': log.las_member,
        'source_xhd_member': log.xhd_member,
        'source_xrd_member': log.xrd_member,
        'classification': classify_log(log),
        'log_created': str(log.log_created) if log.log_created is not None else None,
        'view_type': log.view_type,
        'index_unit': log.index_unit,
        'well_header': log.well,
        'sonde_stack': log.xhd.get('selected_stack', []),
        'curves': [curve.__dict__ for curve in log.curves],
        'warnings': log_qc_issues(log),
    }


def build_caliper_app_table(geometry: pd.DataFrame, hole_id: str) -> pd.DataFrame:
    columns = [
        'depth_m', 'X1', 'X2', 'Y1', 'Y2',
        'diameter_x_mm', 'diameter_y_mm', 'equivalent_diameter_mm',
        'major_diameter_mm', 'minor_diameter_mm', 'ovality_pct',
        'hole_centre_x_mm', 'hole_centre_y_mm', 'tool_offset_mm',
        'area_m2', 'segment_volume_m3', 'cumulative_volume_m3',
        'segment_nominal_volume_m3', 'cumulative_nominal_volume_m3',
        'segment_volume_variance_m3', 'cumulative_volume_variance_m3',
    ]
    output = geometry[[column for column in columns if column in geometry]].copy()
    output.insert(0, 'hole_id', hole_id)
    output['geometry_mode'] = geometry.attrs.get('geometry_mode')
    return output

caliper_app_samples = build_caliper_app_table(caliper_geometry, calibrated_id)
metadata_payload = build_log_metadata_payload(calibrated_log)

display(caliper_app_samples.head())
display(pd.DataFrame([{
    'table': 'hole_log_samples',
    'grain': 'one row per sampled depth',
    'purpose': 'curves, derived caliper geometry, area and segment volume',
}, {
    'table': 'hole_log_metadata',
    'grain': 'one row/document per log',
    'purpose': 'provenance, headers, sonde stack, calibration and QC',
}, {
    'table': 'hole_plan',
    'grain': 'one row per planned hole',
    'purpose': 'planned depth, diameter, collar and trajectory',
}, {
    'table': 'hole_reconciliation',
    'grain': 'one row per inspected hole/run',
    'purpose': 'variance metrics, tolerance status and review workflow',
}]))

DO_EXPORT = False
if DO_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    summary_table.to_csv(EXPORT_DIR / 'wireline_log_inventory.csv', index=False)
    archive_inventory.to_csv(EXPORT_DIR / 'archive_companion_inventory.csv', index=False)
    caliper_app_samples.to_csv(EXPORT_DIR / f'{calibrated_id}_caliper_samples.csv', index=False)
    register_reconciliation.to_csv(EXPORT_DIR / f'{calibrated_id}_reconciliation.csv', index=False)
    interval_summary.to_csv(EXPORT_DIR / f'{calibrated_id}_interval_summary.csv', index=False)
    with (EXPORT_DIR / f'{calibrated_id}_metadata.json').open('w', encoding='utf-8') as handle:
        json.dump(metadata_payload, handle, indent=2, default=str)
    if trajectory is not None:
        trajectory.to_csv(EXPORT_DIR / f'{trajectory.attrs.get("log_id")}_trajectory.csv', index=False)
    print(f'Exports written to: {EXPORT_DIR.resolve()}')
else:
    print('Set DO_EXPORT = True to write CSV/JSON app hand-off files.')

,hole_id,depth_m,X1,X2,Y1,Y2,diameter_x_mm,diameter_y_mm,equivalent_diameter_mm,major_diameter_mm,minor_diameter_mm,ovality_pct,hole_centre_x_mm,hole_centre_y_mm,tool_offset_mm,area_m2,segment_volume_m3,cumulative_volume_m3,segment_nominal_volume_m3,cumulative_nominal_volume_m3,segment_volume_variance_m3,cumulative_volume_variance_m3,geometry_mode
0,Calibrated,2.0200,61.3744,67.6524,79.5509,72.7905,129.0268,152.3414,140.2003,152.3414,129.0268,16.5723,-3.1390,3.3802,4.6129,0.0154,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,opposite_arm_sum
1,Calibrated,2.0300,61.3753,67.6531,79.5518,72.7903,129.0284,152.3421,140.2015,152.3421,129.0284,16.5715,-3.1389,3.3808,4.6133,0.0154,0.0002,0.0002,0.0002,0.0002,-0.0000,-0.0000,opposite_arm_sum
2,Calibrated,2.0400,61.3761,67.6539,79.5527,72.7900,129.0300,152.3427,140.2026,152.3427,129.0300,16.5707,-3.1389,3.3813,4.6137,0.0154,0.0002,0.0003,0.0002,0.0004,-0.0000,-0.0000,opposite_arm_sum
3,Calibrated,2.0500,61.3770,67.6546,79.5536,72.7898,129.0316,152.3434,140.2038,152.3434,129.0316,16.5699,-3.1388,3.3819,4.6140,0.0154,0.0002,0.0005,0.0002,0.0005,-0.0000,-0.0001,opposite_arm_sum
4,Calibrated,2.0600,61.3779,67.6553,79.5545,72.7895,129.0332,152.3440,140.2050,152.3440,129.0332,16.5691,-3.1387,3.3825,4.6144,0.0154,0.0002,0.0006,0.0002,0.0007,-0.0000,-0.0001,opposite_arm_sum


,table,grain,purpose
0,hole_log_samples,one row per sampled depth,"curves, derived caliper geometry, area and seg..."
1,hole_log_metadata,one row/document per log,"provenance, headers, sonde stack, calibration ..."
2,hole_plan,one row per planned hole,"planned depth, diameter, collar and trajectory"
3,hole_reconciliation,one row per inspected hole/run,"variance metrics, tolerance status and review ..."


Set DO_EXPORT = True to write CSV/JSON app hand-off files.


## 15. Supplied-file regression checks

In [19]:
# These checks are intentionally tied to the supplied fixture archives. They provide a fast signal
# if parser changes alter curve mapping, calibration handling or engineering calculations.
fixture_archive_names = {path.name for path in ZIP_INPUTS}
fixture_mode = {'Caliper(1).zip', 'ACS03 Calibration runs(1).zip'}.issubset(fixture_archive_names)

assert len(logs) >= 1
assert all(len(log.df) > 0 for log in logs.values())
assert all(len(log.curves) == log.df.shape[1] for log in logs.values())
assert len(caliper_ids) >= 1
assert all(calibrated_log.curve(arm) is not None for arm in ARM_NAMES)
assert len(cal_table) == 4
assert set(cal_table['channel']) == set(ARM_NAMES)
assert (cal_table['mode'] == 'ThreePoint').all()
assert caliper_geometry['segment_volume_m3'].sum() > 0
assert caliper_geometry['equivalent_diameter_mm'].notna().all()
assert caliper_summary['integrated_length_m'].iloc[0] > 0

if trajectory is not None:
    assert trajectory['tvd_m'].iloc[-1] >= 0
    assert np.isfinite(trajectory[['easting_offset_m', 'northing_offset_m', 'tvd_m']].to_numpy()).all()

if fixture_mode:
    assert len(logs) == 13, f'Expected 13 supplied LAS files; found {len(logs)}.'
    missing_las = archive_inventory.loc[~archive_inventory['LAS'], 'item'].tolist()
    assert 'ACS-03_Run10_10m_min_up' in missing_las
    assert all(any('year-minute-day' in warning for warning in log.warnings) for log in logs.values())
    assert uncalibrated_log is not None and caliper_looks_raw(uncalibrated_log)

print('All parser, metadata, caliper, volume and trajectory checks passed.')

All parser, metadata, caliper, volume and trajectory checks passed.


## 16. Conclusions and path to the MiningSim web application

### What the supplied files show

- The caliper data was present but easy to miss: it is stored as four independent channels (`X1`, `X2`, `Y1`, `Y2`) in both LAS files.
- `Uncalibrated.las` is raw-count data; `Calibrated.las` is already on a millimetre scale. The XHD file contains the polynomial needed to transform the raw channels.
- The ACS archive contains gamma, temperature/conductivity and verticality channels. Several early calibration runs contain zero or non-physical channels, so curve-level QC must be part of ingestion rather than an afterthought.
- XHD is essential for reliable timestamps, tool ownership, serial numbers and calibration provenance. The LAS `DATE` field should not be trusted for these files.
- XRD should be stored alongside the interpreted data even though this prototype does not decode it.

### Recommended application decomposition

1. **Ingestion service** — ZIP/LAS/XHD validation, checksum, provenance and immutable raw-file storage.
2. **Canonical log model** — hole/run metadata, sonde-owned curves, units, depth basis and QC flags.
3. **Engineering analytics** — caliper geometry, volume, trajectory, run alignment and reconciliation.
4. **Inspection interface** — interactive tracks, cross-sections, 3D views, interval exceptions and approval workflow.
5. **Persistence/API** — separate raw samples, derived samples, plan records and reconciliation outcomes so calculations can be versioned and reproduced.

Before implementing production volume calculations, obtain written confirmation of the four-arm channel convention and the expected treatment of tool body radius / arm offsets. Once confirmed, lock that convention into a versioned calculation profile and retain the profile identifier with every result.